# 👗 Fashion-MNIST Image Classification using CNN

This notebook builds a **Convolutional Neural Network (CNN)** to classify clothing images from the [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset.

**What you will learn:**
- Why CNNs outperform flat ANNs on image data
- How to use `Conv2D`, `BatchNormalization`, `MaxPooling`, and `Dropout` layers
- How to train with `EarlyStopping` to prevent overfitting
- How to evaluate with `.evaluate()` and `.predict()`
- How to visualize training curves, a confusion matrix, and sample predictions
- How to save the trained model for reuse

---
## 🗺️ Roadmap
```
PART 1 — Setup      : Imports, load & visualize data, preprocess
PART 2 — CNN Model  : Build, compile, and train with EarlyStopping
PART 3 — Evaluate   : Loss, accuracy, confusion matrix, sample predictions
PART 4 — Save       : Save the trained model for reuse
```
> ⚡ Press **Shift + Enter** to run each cell. Read the explanation first!

---
# 📦 PART 1 — Setup

## Step 1 — Import Libraries

| Library | Purpose |
|---|---|
| `numpy` | Array operations |
| `matplotlib` | Visualizations |
| `tensorflow / keras` | Build and train the CNN |
| `sklearn` | Confusion matrix |


In [ ]:
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # suppress TF info/warning logs
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import ConfusionMatrixDisplay

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print('✅ All libraries imported!')

## Step 2 — Load & Explore the Dataset

**Fashion-MNIST** contains **70,000 grayscale images** (28×28 pixels) across 10 clothing categories.

| Label | Class | Label | Class |
|---|---|---|---|
| 0 | T-shirt/Top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

We load it directly from `keras.datasets` — no internet scraping needed.
- **Training set:** 60,000 images  
- **Test set:** 10,000 images

In [ ]:
CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal',  'Shirt',   'Sneaker',  'Bag',   'Ankle boot']

print('⏳ Loading Fashion-MNIST ...')
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

print('✅ Dataset loaded!')
print(f'   Train : {X_train_raw.shape}  →  {len(X_train_raw):,} images')
print(f'   Test  : {X_test_raw.shape}   →  {len(X_test_raw):,} images')
print(f'   Pixel range (raw) : [{X_train_raw.min()}, {X_train_raw.max()}]')

## Step 3 — Visualize Sample Images

Always look at your data before building a model. This confirms labels are correct and the images are what we expect.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Fashion-MNIST — Sample Images (one per class)', fontsize=13, fontweight='bold')

for class_idx in range(10):
    # pick first image of each class from training set
    sample_idx = np.where(y_train == class_idx)[0][0]
    axes[0, class_idx].imshow(X_train_raw[sample_idx], cmap='gray')
    axes[0, class_idx].axis('off')
    axes[0, class_idx].set_title(CLASS_NAMES[class_idx], fontsize=8)

    # pick a second sample
    sample_idx2 = np.where(y_train == class_idx)[0][1]
    axes[1, class_idx].imshow(X_train_raw[sample_idx2], cmap='gray')
    axes[1, class_idx].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved → sample_images.png')

## Step 4 — Preprocess

CNNs require two preprocessing steps:

| Step | Why |
|---|---|
| **Normalize** pixels to [0, 1] | Keeps gradients stable during training |
| **Reshape** to (28, 28, 1) | Adds the *channel* dimension CNNs expect |

The flat ANN would take `(28×28) = 784` pixels — losing all spatial info.  
The CNN keeps the **2D grid** intact and learns from it.

In [ ]:
# Normalize to [0.0, 1.0]
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

# Reshape: (N, 28, 28) → (N, 28, 28, 1)  ← channel dimension
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

print('✅ Preprocessing complete!')
print(f'   CNN input shape : {X_train.shape}   ← 28×28 grid, 1 channel')
print(f'   Pixel range     : [{X_train.min():.1f}, {X_train.max():.1f}]')

---
# 🧠 PART 2 — Build & Train the CNN

## Step 5 — Model Architecture

The CNN is organized into **3 convolutional blocks** followed by a **classifier head**.

```
Input (28×28×1)
    │
    ├─ Block 1: Conv2D(32) → BatchNorm → ReLU → MaxPool(2×2) → Dropout(0.2)
    │           Output: 14×14×32
    │
    ├─ Block 2: Conv2D(64) → BatchNorm → ReLU → MaxPool(2×2) → Dropout(0.25)
    │           Output: 7×7×64
    │
    ├─ Block 3: Conv2D(128) → BatchNorm → ReLU → Dropout(0.3)
    │           Output: 7×7×128   ← no pooling, preserve detail
    │
    └─ Head   : Flatten → Dense(128, L2) → Dropout(0.5)
                        → Dense(64,  L2) → Dropout(0.2)
                        → Dense(10, softmax)
```

**Key design choices:**

| Technique | Effect |
|---|---|
| `Conv2D` with `padding='same'` | Detects patterns without shrinking spatial size |
| `BatchNormalization` | Normalizes layer outputs → faster, stable training |
| `MaxPooling2D` | Downsamples feature maps → reduces compute, adds translation invariance |
| `Dropout` | Randomly disables neurons → forces generalization |
| `L2 regularization` | Penalizes large weights → prevents memorization |

In [ ]:
cnn_model = keras.Sequential([

    layers.Input(shape=(28, 28, 1)),

    # ── Block 1: detect simple patterns (edges, lines) ──────────
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),      # normalize → stable training
    layers.Activation('relu'),
    layers.MaxPooling2D(2, 2),        # 28×28 → 14×14
    layers.Dropout(0.2),

    # ── Block 2: detect medium patterns (shapes, curves) ────────
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2, 2),        # 14×14 → 7×7
    layers.Dropout(0.25),

    # ── Block 3: detect complex patterns (clothing features) ─────
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),              # no pooling — preserve 7×7 detail

    # ── Classifier head ──────────────────────────────────────────
    layers.Flatten(),                 # 7×7×128 = 6,272 values
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax'),  # 10 classes

], name='CNN_BatchNorm')

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # integer labels
    metrics=['accuracy']
)

cnn_model.summary()

## Step 6 — Train with Early Stopping

**Early Stopping** monitors `val_loss` and stops training if it doesn't improve for `patience` epochs.
It also restores the weights from the best epoch — so we never accidentally keep an overfit model.

| Setting | Value | Meaning |
|---|---|---|
| `monitor` | `val_loss` | Track validation loss |
| `patience` | `7` | Wait 7 epochs before stopping |
| `restore_best_weights` | `True` | Roll back to the best checkpoint |

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

print('🚀 Training CNN ...')
print('   Conv layers preserve 2D spatial structure → expect strong accuracy!\n')

history = cnn_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

print('\n✅ CNN training complete!')

---
# 📊 PART 3 — Evaluate

## Step 7 — Test Set Performance

We evaluate two ways:

| Method | Returns | Use when |
|---|---|---|
| `.evaluate()` | Loss + accuracy as scalars | Quick pass/fail check |
| `.predict()` | Probability vector per image (shape: N×10) | Need per-class confidence or per-sample labels |

In [ ]:
# Method 1: .evaluate()
test_loss, test_acc = cnn_model.evaluate(X_test, y_test, verbose=0)

print('=' * 42)
print('   CNN — Test Set Results')
print('=' * 42)
print(f'   Test Loss     : {test_loss:.4f}')
print(f'   Test Accuracy : {test_acc * 100:.2f}%')
print('=' * 42)

# Method 2: .predict()
probs = cnn_model.predict(X_test, verbose=0)   # shape: (10000, 10)
preds = probs.argmax(axis=1)                   # predicted class per image

print(f'\n.predict() output shape : {probs.shape}')
print(f'First 10 predicted       : {preds[:10]}')
print(f'First 10 actual          : {y_test[:10]}')
print(f'Manual accuracy          : {(preds == y_test).mean() * 100:.2f}%')

## Step 8 — Training Curves

Plotting training vs validation curves tells you:
- **Train ≈ Val** → healthy generalization
- **Train >> Val** → overfitting (model memorized training data)
- **Both high loss** → underfitting (model too simple)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('CNN Training Progress', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(history.history['loss'],     label='Train',      color='steelblue', lw=2)
axes[0].plot(history.history['val_loss'], label='Validation', color='tomato',    lw=2, linestyle='--')
axes[0].set_title('Loss (Cross-Entropy) over Epochs', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'],     label='Train',      color='steelblue', lw=2)
axes[1].plot(history.history['val_accuracy'], label='Validation', color='tomato',    lw=2, linestyle='--')
axes[1].axhline(test_acc, color='green', linestyle=':', lw=1.5, label=f'Test Acc ({test_acc*100:.1f}%)')
axes[1].set_title('Accuracy over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → training_curves.png')

## Step 9 — Confusion Matrix

A confusion matrix shows **where the model gets confused** between classes.
- **Diagonal cells** = correct predictions
- **Off-diagonal cells** = mistakes (row = true label, column = predicted)

Common confusions in Fashion-MNIST: Shirt vs T-shirt, Coat vs Pullover.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test, preds,
    display_labels=CLASS_NAMES,
    ax=ax,
    colorbar=False,
    xticks_rotation=45
)

ax.set_title(
    f'CNN Confusion Matrix  (Test Accuracy: {test_acc*100:.1f}%)\n'
    'Diagonal = correct  |  Off-diagonal = mistakes',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → confusion_matrix.png')

## Step 10 — Sample Predictions

Visual check of predictions on the first 10 test images.
- 🟢 **Green** = correct prediction
- 🔴 **Red** = wrong prediction

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Sample Predictions  |  Green = correct   Red = wrong',
             fontsize=12, fontweight='bold')

for i in range(10):
    img = X_test[i].reshape(28, 28)
    color = 'green' if preds[i] == y_test[i] else 'red'
    conf  = probs[i, preds[i]] * 100

    # Top row: image
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(CLASS_NAMES[preds[i]], fontsize=8, color=color)

    # Bottom row: confidence bar
    axes[1, i].bar(range(10), probs[i], color='steelblue', alpha=0.7)
    axes[1, i].bar(preds[i], probs[i, preds[i]], color=color)
    axes[1, i].set_xticks([])
    axes[1, i].set_yticks([])
    axes[1, i].set_title(f'{conf:.0f}%', fontsize=8)

axes[0, 0].set_ylabel('Image',      fontsize=10, labelpad=6)
axes[1, 0].set_ylabel('Confidence', fontsize=10, labelpad=6)

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → sample_predictions.png')

---
# 💾 PART 4 — Save the Model

## Step 11 — Save & Reload

`model.save()` stores the full model — architecture, weights, and compile settings — so you can reload it anytime without retraining.

```python
# To reload later:
loaded_model = keras.models.load_model('fashion_cnn_model.keras')
```

In [ ]:
MODEL_PATH = 'fashion_cnn_model.keras'
cnn_model.save(MODEL_PATH)
print(f'✅ Model saved → {MODEL_PATH}')

# Verify reload
loaded = keras.models.load_model(MODEL_PATH)
reload_loss, reload_acc = loaded.evaluate(X_test, y_test, verbose=0)
print(f'✅ Reload verified — accuracy: {reload_acc * 100:.2f}%  (matches original)')

---
## 🎉 Summary

### CNN Architecture

| Layer Block | Filters | Output Size | Purpose |
|---|---|---|---|
| Conv Block 1 | 32 | 14×14×32 | Detect edges and lines |
| Conv Block 2 | 64 | 7×7×64 | Detect shapes and curves |
| Conv Block 3 | 128 | 7×7×128 | Detect complex clothing features |
| Dense Head | 128 → 64 → 10 | — | Classify into 10 categories |

### Regularization Techniques Used

| Technique | Effect |
|---|---|
| `BatchNormalization` | Stabilizes and speeds up training |
| `Dropout` | Prevents overfitting by disabling random neurons |
| `L2 Regularization` | Penalizes large weights in dense layers |
| `EarlyStopping` | Stops training before the model overfits |

### Why CNN beats a Flat ANN on Images
- A flat ANN destroys the 2D structure by flattening pixels into a vector
- CNN's `Conv2D` layers slide learned filters across the 2D image, detecting local patterns regardless of where they appear (**translation invariance**)
- Each successive block detects increasingly complex patterns: edges → shapes → features

---
> 💡 **Want to go further?** Try adding a 4th Conv block, using `GlobalAveragePooling2D` instead of `Flatten`, or experimenting with learning rate schedules.